Dans ce notebook on va séparer en train et test nos données avant analyse préalable de la base pour éviter toute forme de data leakage. 

In [1]:
import os 
import pandas as pd 
from sklearn.model_selection import train_test_split
import numpy as np
import plotly.graph_objects as go

In [2]:
print(os.getcwd())
os.chdir("../")
os.getcwd()

c:\Users\leoco\Documents\cours\M2_MOSEF\ML_theory\land_value_prediction\notebooks


'c:\\Users\\leoco\\Documents\\cours\\M2_MOSEF\\ML_theory\\land_value_prediction'

In [3]:
from src.land_value_prediction.train_test_split.analysis_functions import comparer_train_test, identifier_differences_significatives

In [4]:
idf_vf_full = pd.read_parquet("data/processed/idf_vf_full.parquet")
idf_vf_full.head()

,date_mutation,nature_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,...,commune_part_admin_sante,commune_etablissements_par_menage,commune_taux_etablissements_10_plus,commune_sante_score_2013_commune,commune_education_score_2013_commune,commune_revenu_score_2013_commune,commune_idh2_2013_commune,commune_taux_criminalite_moyen,commune_taux_croissance_pop,commune_densite_pop
0,2021-01-01,Vente,169500.0,ALL HOCHE,92130.0,92040,Issy-les-Moulineaux,92,2,2.0,...,0.090161,0.075901,0.254014,0.700824,0.835437,0.704796,0.747019,0.376209,1.045371,1.610612e+10
1,2021-01-02,Vente,512000.0,RUE DE CHARONNE,75011.0,75111,Paris 11e Arrondissement,75,1,2.0,...,0.067020,0.131609,0.145909,0.680558,0.851424,0.607516,0.713166,0.996164,-1.129705,3.995722e+10
2,2021-01-02,Vente,185000.0,AV GAL LECLERC,95250.0,95051,Beauchamp,95,1,2.0,...,0.052632,0.088357,0.207430,0.744042,0.506516,0.626639,0.625732,0.380534,-0.311392,2.875166e+09
3,2021-01-02,Vente,415000.0,RUE DES COURLIS,95100.0,95018,Argenteuil,95,0,1.0,...,0.085686,0.071492,0.182996,0.567644,0.364296,0.331115,0.421018,0.561570,0.784827,6.400116e+09
4,2021-01-02,Vente,415000.0,RUE DES COURLIS,95100.0,95018,Argenteuil,95,0,1.0,...,0.085686,0.071492,0.182996,0.567644,0.364296,0.331115,0.421018,0.561570,0.784827,6.400116e+09


On va créer 3 échantillons :  
- échantillons train et test sur 2021-2024  
- échantillon temporel out_of_sample (2025-S1)  

Séparation train-test et échantillon out of sample :

In [5]:
train_test = idf_vf_full[idf_vf_full['annee'] < 2025].copy()
out_of_sample_df = idf_vf_full[idf_vf_full['annee'] >= 2025].copy()

print(f"Taille du train-test: {len(train_test)} ({len(train_test)/len(idf_vf_full)*100:.1f}%)")
print(f"Taille du out of sample: {len(out_of_sample_df)} ({len(out_of_sample_df)/len(idf_vf_full)*100:.1f}%)")
print(f"\nPériode train-test: {train_test['date_mutation'].min()} à {train_test['date_mutation'].max()}")
print(f"Période out of sample: {out_of_sample_df['date_mutation'].min()} à {out_of_sample_df['date_mutation'].max()}")

print("\nStatistiques prix_m2 - Train-test:")
train_test['prix_m2'].describe().round(2)

Taille du train-test: 661739 (92.9%)
Taille du out of sample: 50806 (7.1%)

Période train-test: 2021-01-01 00:00:00 à 2024-12-31 00:00:00
Période out of sample: 2025-01-02 00:00:00 à 2025-06-30 00:00:00

Statistiques prix_m2 - Train-test:


count    661739.00
mean       5988.05
std        3586.43
min         500.24
25%        3327.13
50%        4800.00
75%        8064.52
max       19998.89
Name: prix_m2, dtype: float64

In [6]:
print("\nStatistiques prix_m2 - Out of sample:")
out_of_sample_df['prix_m2'].describe().round(2)


Statistiques prix_m2 - Out of sample:


count    50806.00
mean      5823.52
std       3529.33
min        504.00
25%       3196.65
50%       4683.54
75%       7888.89
max      19988.70
Name: prix_m2, dtype: float64

Séparer train et test :

In [7]:
# Créer des déciles sur la variable cible pour stratifier le split
idf_vf_full['prix_m2_bins'] = pd.qcut(idf_vf_full['prix_m2'], q=10, labels=False, duplicates='drop')

# Split train/test avec stratification
train, test = train_test_split(
    idf_vf_full, 
    test_size=0.2, 
    shuffle=True,
    random_state=42, 
    stratify=idf_vf_full['prix_m2_bins']
)

# Supprimer la colonne temporaire de bins
train = train.drop('prix_m2_bins', axis=1)
test = test.drop('prix_m2_bins', axis=1)

print(f"Taille du train set: {len(train)} ({len(train)/len(idf_vf_full)*100:.1f}%)")
print(f"Taille du test set: {len(test)} ({len(test)/len(idf_vf_full)*100:.1f}%)")

print("\nStatistiques prix_m2 - Train:")
train['prix_m2'].describe()

Taille du train set: 570036 (80.0%)
Taille du test set: 142509 (20.0%)

Statistiques prix_m2 - Train:


count    570036.000000
mean       5976.599396
std        3582.705371
min         500.240000
25%        3315.430885
50%        4791.666667
75%        8053.333333
max       19998.888889
Name: prix_m2, dtype: float64

In [8]:
print("\nStatistiques prix_m2 - Test:")
test['prix_m2'].describe()


Statistiques prix_m2 - Test:


count    142509.000000
mean       5975.183523
std        3582.366350
min         500.240000
25%        3314.606742
50%        4791.666667
75%        8043.478261
max       19995.454545
Name: prix_m2, dtype: float64

In [9]:
comparison = comparer_train_test(train, test)
comparison

,type,train_mean,test_mean,diff_mean,train_median,test_median,diff_median,train_std,test_std,diff_std,train_min,test_min,train_max,test_max,diff_mean_pct,diff_median_pct,diff_std_pct
variable,,,,,,,,,,,,,,,,,
date_mutation,categorical,<NA>,<NA>,<NA>,2025-03-31 00:00:00,2025-03-31 00:00:00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
nature_mutation,categorical,<NA>,<NA>,<NA>,Vente,Vente,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
valeur_fonciere,numeric,390662.797933,391476.286025,-813.488091,299000.0,300000.0,-1000.0,345126.796452,348479.178198,-3352.381745,6000.0,5200.0,9639800.0,10500000.0,-0.2078,-0.333333,-0.962003
adresse_nom_voie,categorical,<NA>,<NA>,<NA>,RUE DE PARIS,RUE DE PARIS,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
code_postal,numeric,85703.513762,85746.069582,-42.555819,91280.0,91290.0,-10.0,8365.604498,8355.301456,10.303042,75001.0,75001.0,95880.0,95880.0,-0.04963,-0.010954,0.123311
code_commune,categorical,<NA>,<NA>,<NA>,75115,75115,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
nom_commune,categorical,<NA>,<NA>,<NA>,Paris 15e Arrondissement,Paris 15e Arrondissement,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
code_departement,categorical,<NA>,<NA>,<NA>,75,75,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
nombre_lots,numeric,1.080665,1.082311,-0.001646,1.0,1.0,0.0,0.942041,0.942335,-0.000294,0,0,23,18,-0.152037,0.0,-0.031198


In [10]:
warnings = identifier_differences_significatives(comparison, threshold_mean=0.1, threshold_std=0.2)

# Afficher les alertes
if warnings['mean']:
    print("⚠️ Variables avec différences de moyenne > 10%:")
    for w in warnings['mean']:
        print(f"  - {w['variable']}: {w['diff_pct']:.1f}%")

if warnings['std']:
    print("\n⚠️ Variables avec différences d'écart-type > 20%:")
    for w in warnings['std']:
        print(f"  - {w['variable']}: {w['diff_pct']:.1f}%")

if warnings['distribution']:
    print("\n⚠️ Variables avec plage de valeurs différentes:")
    for w in warnings['distribution']:
        print(f"  - {w['variable']}: train={w['train_range']}, test={w['test_range']}")


⚠️ Variables avec différences d'écart-type > 20%:
  - surface_terrain: -50.0%

⚠️ Variables avec plage de valeurs différentes:
  - valeur_fonciere: train=[6000.0, 9639800.0], test=[5200.0, 10500000.0]
  - surface_terrain: train=[1.0, 181873.0], test=[1.0, 448021.0]
  - longitude: train=[1.459688, 3.518413], test=[1.453111, 3.487387]


In [11]:
fig = go.Figure()

# Ajouter les distributions
fig.add_trace(go.Box(
    y=train['prix_m2'],
    name='Train',
    boxmean='sd',
    marker_color='lightblue'
))

fig.add_trace(go.Box(
    y=test['prix_m2'],
    name='Test',
    boxmean='sd',
    marker_color='lightgreen'
))

fig.add_trace(go.Box(
    y=out_of_sample_df['prix_m2'],
    name='Out of Sample (2025)',
    boxmean='sd',
    marker_color='lightcoral'
))

# Mise en forme
fig.update_layout(
    title='Comparaison de la distribution de prix_m2 entre les échantillons',
    yaxis_title='Prix au m² (€)',
    showlegend=True,
    height=600,
    template='plotly_white'
)

fig.show()

In [12]:
train.to_parquet("data/processed/train_test_out_sample_split/idf_vf_train.parquet", index=False)
test.to_parquet("data/processed/train_test_out_sample_split/idf_vf_test.parquet", index=False)
out_of_sample_df.to_parquet("data/processed/train_test_out_sample_split/idf_vf_out_of_sample.parquet", index=False)